# PoC: RAG Sistemlerinde Retrieval Quality ve Chunk Attribution Analizi

**Hazırlayan:** Duygu Can   

---

Bu PoC çalışmasının temel amacı, OpenAI GPT-4o dil modeli ve Google Gemini Embedding (models/embedding-001) altyapısı üzerine kurgulanan RAG mimarisinde döküman parçalarının cevap üretimine olan etkisini ölçen kapsamlı bir değerlendirme çerçevesi geliştirmektir. Çalışma kapsamında LangChain kütüphanesi ile uygulanan farklı parçalama stratejilerinin erişim performansı, FAISS vektör veritabanı üzerinde test edilerek "Retrieval Quality" analiz edilmekte, üretilen cevapların doğruluğu ve kaynak sadakati ise Attribution mekanizmalarıyla teyit edilmektedir.

In [ ]:
import os
import re
import json
import time
import random
import warnings
from datetime import datetime
import pickle
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi  # Lexical matching
import faiss
from tqdm.notebook import tqdm

# Suppress Pydantic/LangChain version warnings
warnings.filterwarnings("ignore")

# Load environment variables
from dotenv import load_dotenv
load_dotenv()

# Set User Agent for LangChain loaders
os.environ["USER_AGENT"] = "RAG_PoC_Chunking/1.0"

from google import genai 
google_api_key = os.getenv("GOOGLE_API_KEY")
google_client = genai.Client(api_key=google_api_key) 

from openai import OpenAI
openai_api_key = os.getenv("OPENAI_API_KEY")
openai_client = OpenAI(api_key=openai_api_key)

import bs4
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

## 1. Veri Seti ve Araç Seçimi

### 1.1. Döküman Seti Seçimi:

Bu çalışmada veri seti olarak, zengin içerik çeşitliliği ve yüksek teknik derinliği nedeniyle Scikit-learn Teknik Dokümantasyonu (User Guide) tercih edilmiştir. Bu döküman setinin seçilme gerekçesi, sunduğu karmaşık kavramlar ve hiyerarşik yapı sayesinde, dil modelinin bilgiyi geri getirme (retrieval) ve kavramlar arası ilişkilendirme yeteneğini test etmek için ideal bir manifold sunmasıdır. Projenin teknoloji yığını ise; döküman işleme ve pipeline yönetimi için LangChain kütüphanesi, hızlı ve verimli vektör arama işlemleri için FAISS vektör veritabanı ve yüksek muhakeme kapasitesine sahip Gemini 2.0/1.5 Flash modellerinden oluşmaktadır.

In [2]:
# Data ingestion
loader = PyPDFLoader("data/scikit-learn-docs.pdf")
docs = loader.load()

# Basic verification 
total_chars = sum(len(doc.page_content) for doc in docs)

print(f"Document loaded successfully.")
print(f"Total Pages: {len(docs)}")
print(f"Total Character Count: {total_chars}")

Document loaded successfully.
Total Pages: 2503
Total Character Count: 4796571


### 1.2. Chunking Stratejileri

Kullanılan veri seti, yoğun kod blokları ile teorik açıklamaları bir arada barındıran hibrit bir yapıdadır. Pinecone (2023) tarafından yayınlanan 'Chunking Strategies for LLM Applications' rehberindeki standartlara bağlı kalarak, iki farklı uç noktayı temsil eden stratejiler belirlenmiştir. Literatürdeki 1 Token≈4 Karakter kabulü üzerinden yapılan seçimler şöyledir:

1. **Fixed-size Chunking (Baseline / Geniş Bağlam):** Sistemin temel performansını ölçmek amacıyla kullanılan bu stratejide, bağlamı korumaya çalışılmıştır. Pinecone rehberinde, konu bütünlüğünü korumak için 512-1024 token aralığı önerilmektedir. Bu doğrultuda, teorik açıklamaların ve teknik kavramların bütünlüğünü sağlamak adına 2000 karakterlik (~500 token) geniş pencereler tercih edilmiştir.

2. **Document-structure Aware Chunking (Yapısal / Granüler Bilgi):** Metnin topolojisini korumayı hedefleyen bu stratejide ise "Granular Information" yaklaşımı izlenmiştir. Scikit-learn dökümanındaki fonksiyon tanımları ve parametre listeleri gibi detayları yakalamak amacıyla, Pinecone'un 128-256 token önerisinin alt sınırına odaklanılmıştır. Hiyerarşik bir yöntemle; öncelikle paragrafları, sonra satırları ve cümleleri koruyan 500 karakterlik (~125 token) hassas pencereler oluşturulmuştur.

Her iki stratejide de sınır noktalarındaki bilgi kaybını minimize etmek için %10 oranında (sırasıyla 200 ve 50 karakter) örtüşme payı uygulanmıştır. Literatürde Karpukhin ve ekibi (2020), Dense Passage Retrieval çalışmalarında 100 kelimelik ayrık bloklar kullanmış ve örtüşen pasajların performansa belirgin bir katkı sağlamadığını raporlamıştır. Ancak bu çalışmada, özellikle kod bloklarının veya teknik tanımların kesim noktalarında oluşabilecek anlamsal kopukluk riskine karşı, RAG mimarilerinde "güvenli bir liman" olarak kabul edilen örtüşme stratejisine sadık kalınmıştır.

In [3]:
# Fixed-size Strategy for baseline: larger, mechanical parts
fixed_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200,
    separators=[" "] # split from spaces
)
fixed_chunks = fixed_splitter.split_documents(docs)

# 2. Structure-aware Strategy for comparision: smaller, atomic pieces
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " ", ""]  # Maintain the hierarchy
)
struct_chunks = recursive_splitter.split_documents(docs)

print(f"Fixed-size Strategy (2000ch): {len(fixed_chunks)} chunks")
print(f"Structure-aware Strategy (500ch): {len(struct_chunks)} chunks")

Fixed-size Strategy (2000ch): 3722 chunks
Structure-aware Strategy (500ch): 11696 chunks


Değerlendirme sonuçlarının kaynak dokümanla olan izlenebilirliğini sağlamak ve farklı chunking metotlarının başarısını kıyaslamak için her chunka benzersiz bir ID ve strateji etiketi atanmıştır. 

In [4]:
def process_and_save_chunks(chunks, prefix, strategy_name, filename):
    """
    Prints the ID and strategy label to the chunk list, then saves it as JSON.
    
    Args:
        chunks (list): List of LangChain Document objects.
        prefix (str): Prefix for the ID (fix or struct).
        strategy_name (str): Strategy name (fixed or recursive).
        filename (str): File path to save.
    """
    # Folder control
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    
    json_data = []
    
    print(f"Processing: {strategy_name} ({len(chunks)} chunks)...")

    for i, chunk in enumerate(chunks):
        # ID & strategy assignment
        current_id = f"{prefix}_{i}"
        chunk.metadata["chunk_id"] = current_id
        chunk.metadata["strategy"] = strategy_name
        
        # Prepare the data
        entry = {
            "chunk_id": current_id,
            "strategy": strategy_name,
            "content": chunk.page_content,
            "metadata": chunk.metadata  # to be saved
        }
        json_data.append(entry)
    
    # Saving
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(json_data, f, ensure_ascii=False, indent=2)
        
    print(f"Done: Saved to the ‘{filename}’ file.")

# Output directory
output_dir = "data/processed"
os.makedirs(output_dir, exist_ok=True)


# Define tasks for both chunking strategies
chunk_tasks = [
    {
        "chunks": fixed_chunks,
        "prefix": "fix",
        "strategy_name": "fixed-size",
        "filename": os.path.join(output_dir, "fixed_chunks.json")
    },
    {
        "chunks": struct_chunks,
        "prefix": "struct",
        "strategy_name": "structure-aware",
        "filename": os.path.join(output_dir, "struct_chunks.json")
    }
]

# Process each chunking strategy
for task in chunk_tasks:
    # Check if file already exists
    if os.path.exists(task["filename"]):
        print(f"Skipping: {task['filename']} already exists.")
    else: # if not, process and save
        print(f"Processing: Creating {task['filename']}...")
        process_and_save_chunks(
            chunks=task["chunks"],
            prefix=task["prefix"],
            strategy_name=task["strategy_name"],
            filename=task["filename"]
        )

Skipping: data/processed\fixed_chunks.json already exists.
Skipping: data/processed\struct_chunks.json already exists.


### 1.3. Attribution Yöntemleri

Modelin ürettiği cevabın, getirilen döküman parçasına ne kadar sadık kaldığını ölçmek için üç farklı metrik kurgulanmıştır.

##### 1.3.1. Lexical Overlap 

Cevap ve kaynak metin arasındaki ortak token'ların kesişim kümesini ölçen bu metrik, sistemin kaynağa sadık kalıp kalmadığını ve "halüsinasyon" görüp görmediğini denetlemek amacıyla kullanılmaktadır. Modelin, kaynak dökümanda geçen teknik terimleri doğru kullanması durumunda bu skorun yüksek çıkması beklenir.

Hesaplama maliyetinin düşük olması, hızı ve sonuçlarının şeffaf/açıklanabilir olması bu yöntemin tercih edilmesindeki ana etkenlerdir. Ancak yöntemin temel kısıtı, "eş anlamlılık" durumlarını tespit edememesidir. Örneğin, Scikit-learn bağlamında metinde "column" (sütun) terimi geçerken modelin cevapta "feature" (nitelik) terimini kullanması durumunda, anlamsal bütünlük tam olsa dahi Lexical skor 0 üretilecektir. Bu durum, metrikte False Negative riskini artırmaktadır.

In [5]:
def calculate_lexical_score(text1, text2):
    """
    Calculates the word overlap ratio (Jaccard Index) between two texts.
    Args:
        text1 (str): First text.
        text2 (str): Second text.
    Returns:
        float: Jaccard similarity score between 0 and 1.
    """

    # Clean the text and convert it to a word set
    tokens1 = set(re.findall(r'\w+', text1.lower()))
    tokens2 = set(re.findall(r'\w+', text2.lower()))
    
    if not tokens1 or not tokens2:
        return 0.0
    
    intersection = tokens1.intersection(tokens2)
    union = tokens1.union(tokens2)
    
    # Jaccard Score: Intersection / Union
    return len(intersection) / len(union)

##### 1.3.2. Semantic Similarity

Bu metrik, model cevabını ve kaynak metni yüksek boyutlu vektör uzayına izdüşürerek aralarındaki kosinüs benzerliğini hesaplar. Böylece sözcük bazlı eşleşme olmasa dahi anlamsal doğruluğu ölçmeyi hedefler. Örneğin, Scikit-Learn’deki 'standardize' ve 'normalize' terimleri matematiksel olarak farklı işlemleri ifade etse de, veri işleme bağlamında anlamsal olarak birbirlerine çok yakın olduklarından bu yöntemle Lexical Overlap'e göre daha başarılı şekilde ilişkilendirilirler. Ancak yüksek hesaplama maliyeti ve farklı bağlamdaki ifadelerin vektör uzayında birbirine yakın konumlanmasından kaynaklanan False Positive riski vardır.

Sistemin anlamsal motoru için başlangıçta MTEB (Massive Text Embedding Benchmark) liderlik tablosunda "Code Information Retrieval" alanında optimize edilmiş Qwen3-Embedding-4B modeli değerlendirilmiştir (Li et al., 2024). Sadece İngilizce ve Python dillerini kapsayacak şekilde özelleştirilen benchmark sonuçlarına göre bu model, top 5'deki rakipleri ile yarışan %100 Zero-shot kabiliyeti ve %79.20 retrieval başarısıyla yüksek doğruluk sunarken, 4B parametrelik yapısı ve 2560 boyutlu vektör çıktısıyla daha verimlidir. Ancak hızlı prototipleme için, yerel kurulum karmaşıklığından kaçınmak için gemini-embedding-001 seçilmiştir (Google DeepMind, 2024).. Bu model, yine özelleştirilmiş MTEB lider listesinde 6. sırada yer alması, %100 Zero-shot başarımı ve %73.90'lık güçlü retrieval skoruyla rekabetçi bir performans göstermektedir.

Metrik implementasyonunda negatif kosinüs değerlerinin sıfıra çekilmesi (ReLU yaklaşımı), vektör uzayında "zıt anlam" (-1) ile "ilgisizlik" (0) durumlarının RAG bağlamında eşdeğer kabul edilmesine dayanır. RAG mimarisinde dökümanlar, üretilen cevaba ya anlamsal destek sağlar ya da alakasız kalarak etkisiz kalır. Bir dökümanın cevabı matematiksel olarak 'negatif' etkilemesi anlamsal olarak karşılık bulmadığı için, negatif benzerlik değerleri sistemin gürültüden arındırılması adına sıfıra normalize edilmiştir. Yapılan normalizasyon, metriği 0-1 aralığına sabitleyerek sonucu  yüzdelik başarı olarak okunabilen, net bir attribution skoruna dönüştürür.

In [7]:
def calculate_semantic_score(reference_text, generated_text):
    """
    Measures semantic similarity using the latest Gemini Embedding model (001).
    Returns a normalized score between 0.0 (no match) and 1.0 (perfect match).
    Args:
        reference_text (str): The reference text.                   
        generated_text (str): The generated text to compare.
    Returns:            
        float: Normalized semantic similarity score between 0.0 and 1.0.
    """
    try:
        # Generate Embeddings
        result = google_client.models.embed_content(
            model="models/gemini-embedding-001",
            contents=[reference_text, generated_text],
            config={'task_type': 'SEMANTIC_SIMILARITY'} 
        )
        
        # Extract and reshape vectors
        emb_ref = np.array(result.embeddings[0].values).reshape(1, -1)
        emb_gen = np.array(result.embeddings[1].values).reshape(1, -1)
        
        # Calculate Cosine Similarity
        similarity = cosine_similarity(emb_ref, emb_gen)[0][0]
        
        # Apply ReLU-based Normalization
        return float(max(0.0, similarity))
        
    except Exception as e:
        print(f"Embedding error: {e}")
        return 0.0

#### 1.3.3. Ensemble Yaklaşım

Scikit-learn gibi teknik terimlerin yoğun olduğu dökümantasyonlarda, sadece anlama odaklanmak yeterli olmayabilir. Bu nedenle ensemble bir yaklaşım olarak, Dense Passage Retrieval (DPR) makalesinin önerdiği gibi kelimelerin birebir eşleşmesini sağlayan BM25 ile derin anlam ilişkilerini yakalayan Dense Embedding yöntemlerinin ağırlıklı linear kombinasyonu kullanılmıştır (Karpukhin, et al., 2020). Negatif değerleri filtreleyen ReLU mantığı ve normalizasyon süreçleriyle desteklenen bu yaklaşım, dökümanın cevaba olan katkısını 0 ile 1 aralığında, matematiksel olarak doğrulanabilir ve kolayca yorumlanabilir bir attribution skoru haline getirir.

In [6]:
def calculate_ensemble_score(query, reference_chunk, generated_answer, lambda_weight=1.1):
    """
    Calculates a hybrid attribution score combining lexical (BM25) and semantic (Gemini) similarity.
    
    Args:
        query (str): The original user question.
        reference_chunk (str): The retrieved document segment from the knowledge base.
        generated_answer (str): The response produced by the LLM.
        lambda_weight (float): Weighting factor for the semantic score (DPR recommendation: 1.1).
        
    Returns:
        float: A normalized attribution score between 0.0 and 1.0.
    """
    try:
        # Lexical Score (BM25 / Overlap):
        # Tokenizing for a basic lexical match simulation
        tokenized_query = query.lower().split()
        tokenized_chunk = reference_chunk.lower().split()
        bm25 = BM25Okapi([tokenized_chunk])
        lexical_score = bm25.get_scores(tokenized_query)[0]
        
        # Normalizing Lexical Score (Min-Max approximation for the PoC)
        # In a production environment, this should be scaled against a corpus baseline.
        norm_lexical = 1.0 / (1.0 + np.exp(-lexical_score)) 

        # Semantic Score (Gemini-Embedding-001):
        result = google_client.models.embed_content(
            model="models/gemini-embedding-001",
            contents=[reference_chunk, generated_answer],
            config={'task_type': 'SEMANTIC_SIMILARITY'}
        )
        
        emb_ref = np.array(result.embeddings[0].values).reshape(1, -1)
        emb_gen = np.array(result.embeddings[1].values).reshape(1, -1)
        
        raw_similarity = cosine_similarity(emb_ref, emb_gen)[0][0]
        
        # ReLU Logic: Clipping negative similarity to 0.0 to eliminate semantic noise
        semantic_score = float(max(0.0, raw_similarity))

        # Weighted Ensemble (DPR Logic):
        # Combining scores: Attribution = Norm(Lexical) + (Lambda * ReLU(Semantic))
        hybrid_score = (norm_lexical + (lambda_weight * semantic_score)) / (1 + lambda_weight)
        
        return round(float(hybrid_score), 4)

    except Exception as e:
        print(f"Attribution Calculation Error: {e}")
        return 0.0

## 2. Ground Truth QA Seti Oluşturulması

RAG mimarisinin performansını nesnel değerlendirebilmek için, Dense Passage Retrieval (DPR) çalışması (Karpukhin et al., 2020) ve Pinecone ve OpenAI'ın en iyi uygulama standartları (Pinecone, 2023 & OpenAI, n.d.) referans alınarak bir "Ground Truth" soru-cevap veri seti oluşturulmuştur. Sisteminin performansını bütüncül bir yaklaşımla ölçmek amacıyla çok katmalı sorgu stratejisi benimsenmiştir. Dökümandan rassal seçilen her chunk için, sistemin farklı yetkinliklerini hedefleyen üç kategoride soru üretilmiştir. İlk aşamada, sistemin spesifik olguları ve parametreleri yüksek hassasiyetle arama hassasiyetini ölçmek için "Look-up" soruları; dağınık bilgileri sentezleme ve karşılaştırma yeteneklerini test etmek için "Reasoning" soruları kurgulanmıştır. Ek olarak, sistemin anahtar kelime eşleşmesi bulunsa dahi bağlamda yer almayan bilgiye karşı direnç gösterip "Bilmiyorum" diyebilme yeteneğini, halüsinasyon direncini test etmek amacıyla "Hard Negative" örnekler sürece dahil edilmiştir. Kurgulanan bu üçlü yapı sayesinde, sistemin sadece doğruyu bulma yeteneği değil, aynı zamanda yanlıştan da kaçınma güvenilirliğin de sınanması hedeflenmiştir.

Sentetik soru ve "Hard Negative" örneklerin üretiminde temel model olarak GPT-4o kullanılmıştır. Bu tercihin arkasında literatür destekli gerekçeler bulunmaktadır. Zheng ve arkadaşları (2023), "LLM-as-a-Judge" çalışmalarında, yüksek kapasiteli modellerin karmaşık talimatları insanlara denk takip edebildiğini göstermiştir. Bu bulguyla paralel olarak, OpenAI'ın (n.d.) "Evaluation Best Practices" rehberi de sentetik test verisi üretiminde GPT-4 serisi gibi state-of-the-art modellerin kullanılmasını önermektedir. Ayrıca, modelin karmaşık sorgularda bağlamı anlama ve çıkarım yapmadaki üstünlüğü, Sanders ve Heaton (2022) tarafından yürütülen "Question Answering Using Embeddings" deneyleriyle desteklenmiştir. Tüm bu kanıtlar ışığında GPT-4o ile soru-cevap veri kümesi üretilmesine karar verilmiştir.

Veri seti üretim sürecinde, kaynak metni sabit boyutlu parçalamak yerine "Structure-Aware Chunking" stratejisi tercih edilmiştir. Bu seçimin temel nedeni, keyfi karakter sınırlarının yarattığı bilgi kaybını önlemek ve dokümanın anlamsal bütünlüğünü korumaktır. Yüksek kaliteli, tutarlı ve halüsinasyon riski minimize edilmiş bir değerlendirme seti üretebilmek için; modelin işlediği metin parçalarının bağlamsal olarak kesintiye uğramamış, kendi kendine yeten bilgi bloklarından oluşması gerekmektedir.

Modelin stokastik davranışını minimize ederek deterministik çıktı üretmesini sağlamak amacıyla "Temperature" parametresi sıfıra sabitlenmiştir. Bu tercihin temel nedeni, yaratıcı metin üretiminden ziyade, kaynak dokümana sıkı sıkıya bağlı ve halüsinasyondan arındırılmış bir "Ground Truth" seti oluşturma gerekliliğidir.

Üretilen veri setinin güvenilirliği, rastgele seçilen örneklerin insan denetimi ile manuel olarak incelenmesi ve mantıksal tutarlılık kontrollerinin Gemini 3 yapay zeka asistanı desteğiyle çapraz doğrulanması suretiyle teyit edilmiştir.

In [7]:
def generate_qa(chunks, num_chunks_to_process=10):
    """
    Generates a 30-question Ground Truth evaluation set using GPT-4o.
    Produces 3 distinct QA types per chunk (Lookup, Reasoning, Hard Negative) via JSON parsing.
    Args:
        chunks (list): List of LangChain Document objects.
        num_chunks_to_process (int): Number of chunks to process for QA generation.
    Returns:
        pd.DataFrame: DataFrame containing generated QA pairs.

    """
    qa_data = []
    
    # Chunk Selection: Filter chunks > 300 chars for reasoning context
    viable_chunks = [c for c in chunks if len(c.page_content) > 300]
    
    # Select random sample
    sample_size = min(num_chunks_to_process, len(viable_chunks))
    selected_chunks = random.sample(viable_chunks, sample_size)
    
    print(f"Processing {sample_size} chunks using GPT-4o...")
    print(f"Target: {sample_size} chunks processed -> Total {sample_size * 3} questions generated.")

    for i, chunk in enumerate(selected_chunks):
        chunk_id = chunk.metadata.get("chunk_id", "unknown")
        content = chunk.page_content
        
        # The Prompt Structure
        system_msg = "You are a senior expert in creating evaluation datasets for RAG systems."
        
        user_msg = f"""
        Use the following text (Chunk ID: {chunk_id}) to generate evaluation data.

        TEXT:
        {content}

        TASK:
        Generate 3 distinct Question-Answer pairs based SOLELY on the text above.

        RULES:
        1. Pair 1 (Lookup): Specific parameter, default value, or function name.
        2. Pair 2 (Reasoning): A complex question requiring synthesis (e.g., "Why use X?", "Difference between X and Y?").
        3. Pair 3 (Hard Negative): A question containing keywords from the text but NOT answerable by this specific text. The answer MUST be: "I cannot answer this from the given context."
        4. "evidence_quote" field MUST contain the exact substring from the text used to derive the answer. For Hard Negative, this must be null.

        OUTPUT FORMAT (JSON Object):
        {{
          "qa_pairs": [
            {{ "type": "Lookup", "question": "...", "answer": "...", "evidence_quote": "..." }},
            {{ "type": "Reasoning", "question": "...", "answer": "...", "evidence_quote": "..." }},
            {{ "type": "Hard_Negative", "question": "...", "answer": "I cannot answer this from the given context.", "evidence_quote": null }}
          ]
        }}
        """
        
        try:
            # JSON-based response
            response = openai_client.chat.completions.create(
                model="gpt-4o",
                messages=[
                    {"role": "system", "content": system_msg},
                    {"role": "user", "content": user_msg}
                ],
                temperature=0, # Low temperature for consistency
                max_tokens=2000, # safeguard for longer outputs
                response_format={"type": "json_object"}
            )
            
            # Parse response
            raw_json = response.choices[0].message.content
            data = json.loads(raw_json)
            
            # Add to list
            for pair in data["qa_pairs"]:
                qa_data.append({
                    "qa_id": str(uuid.uuid4())[:8],
                    "gold_chunk_id": chunk_id,
                    "model_source": "gpt-4o",
                    "question_type": pair.get("type"),
                    "question": pair.get("question"),
                    "ground_truth_answer": pair.get("answer"),
                    "evidence_quote": pair.get("evidence_quote"),
                    "source_snippet": content[:100] + "..."
                })
            
            print(f"[{i+1}/{sample_size}] Chunk {chunk_id} completed.")
            # Short sleep for rate limiting
            time.sleep(1)
            
        except Exception as e:
            print(f"Error (Chunk {chunk_id}): {e}")
            continue

    return pd.DataFrame(qa_data)

# Define output directory and file path
output_dir = "data/evaluation"
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, "QA_set_gpt4o.xlsx")

# Check if the file already exists to avoid regeneration
if os.path.exists(output_file):
    print(f"File found at {output_file}. Loading existing QA set...")
    df_gpt4_qa = pd.read_excel(output_file)
else:
    print("File not found. Generating new QA set...")
    # Execution
    df_gpt4_qa = generate_qa(struct_chunks, num_chunks_to_process=10)

    # Save to Excel
    df_gpt4_qa.to_excel(output_file, index=False)
    print(f"QA set generated and saved to {output_file}")

# Display summary
print("-" * 50)
print(f"Total number of questions: {len(df_gpt4_qa)}")
print(df_gpt4_qa[["question_type", "question", "ground_truth_answer"]].head())

File found at data/evaluation\QA_set_gpt4o.xlsx. Loading existing QA set...
--------------------------------------------------
Total number of questions: 30
   question_type                                           question  \
0         Lookup  What is the default value of the parameter 'C'...   
1      Reasoning  Why might one choose to use PCA before applyin...   
2  Hard_Negative  What is the purpose of the 'gamma' parameter i...   
3         Lookup  What is the name of the experimental feature t...   
4      Reasoning  Why is it necessary to explicitly import the e...   

                                 ground_truth_answer  
0                                                1.0  
1  Using PCA before applying an SVC in a pipeline...  
2       I cannot answer this from the given context.  
3                      enable_hist_gradient_boosting  
4  The feature is experimental, meaning the predi...  


## 3. RAG Pipeline ve Vector Store Kurulumu

### 3.1. Vector Store Oluşturma

In [9]:
# --- VECTOR STORE CREATION USING GOOGLE GEMINI EMBEDDINGS & FAISS ---
db_path = "data/vector_store"
input_dir = "data/processed"

os.makedirs(db_path, exist_ok=True)
os.makedirs(input_dir, exist_ok=True)

# --- 1. EMBEDDING FUNCTION ---
def get_gemini_embedding(text, task_type="retrieval_document"):
    """
    Wraps the Gemini embedding call with rate limit handling.
    """
    if not text or not isinstance(text, str): return None
    
    max_retries = 5
    wait_time = 4 
    
    for attempt in range(max_retries):
        try:
            result = genai.embed_content(
                model="models/embedding-001",
                content=text,
                task_type=task_type
            )
            time.sleep(wait_time) 
            return np.array(result['embedding'], dtype='float32')
            
        except Exception as e:
            error_msg = str(e).lower()
            if "429" in error_msg or "quota" in error_msg:
                print(f"[WARNING] Quota exceeded. Waiting 30s... (Attempt {attempt+1}/{max_retries})")
                time.sleep(30)
            else:
                print(f"[ERROR] Embedding failed: {e}")
                return None
    return None

# --- 2. VECTOR STORE LOGIC ---
def get_or_create_vector_store(strategy_name):
    """
    Checks if FAISS index exists. If not, creates it.
    """
    index_file = os.path.join(db_path, f"{strategy_name}.faiss")
    metadata_file = os.path.join(db_path, f"{strategy_name}_meta.pkl")
    source_json = os.path.join(input_dir, f"{strategy_name}_chunks.json")

    # A) Check existence
    if os.path.exists(index_file) and os.path.exists(metadata_file):
        print(f"[INFO] Existing store found for '{strategy_name}'. Loading...")
        
        try:
            index = faiss.read_index(index_file)
            with open(metadata_file, "rb") as f:
                metadata_map = pickle.load(f)
            print(f" -> Loaded {index.ntotal} vectors.")
            return index, metadata_map
        except Exception as e:
            print(f"[WARNING] Load failed: {e}. Re-creating...")

    # B) Create New
    print(f"[INFO] Creating new store for '{strategy_name}'...")
    
    if not os.path.exists(source_json):
        print(f"[ERROR] Source file not found: {source_json}")
        return None, None

    with open(source_json, 'r', encoding='utf-8') as f:
        chunks = json.load(f)

    # Initialize FAISS (Dimension 768 for Gemini-001)
    dimension = 768
    index = faiss.IndexFlatIP(dimension)
    metadata_map = {}
    vectors_list = []

    print(f" -> Processing {len(chunks)} chunks...")
    
    for chunk in tqdm(chunks, desc=f"Embedding ({strategy_name})"):
        emb = get_gemini_embedding(chunk['content'], "retrieval_document")
        
        if emb is not None:
            faiss.normalize_L2(emb.reshape(1, -1))
            vectors_list.append(emb)
            
            internal_id = len(vectors_list) - 1
            metadata_map[internal_id] = {
                "chunk_id": chunk['chunk_id'],
                "content": chunk['content'],
                "metadata": chunk.get('metadata', {})
            }

    if vectors_list:
        vectors_np = np.vstack(vectors_list)
        index.add(vectors_np)
        
        faiss.write_index(index, index_file)
        with open(metadata_file, "wb") as f:
            pickle.dump(metadata_map, f)
            
        print(f"[SUCCESS] Saved store to '{index_file}'.")
        return index, metadata_map
    else:
        print("[ERROR] No embeddings generated.")
        return None, None

# --- EXECUTION ---
indices = {}
meta_maps = {}

for strategy in ["fixed", "struct"]:
    idx, meta = get_or_create_vector_store(strategy)
    if idx:
        indices[strategy] = idx
        meta_maps[strategy] = meta

print("-" * 50)
print("Vector Store setup complete.")

[INFO] Creating new store for 'fixed'...
 -> Processing 3722 chunks...


Embedding (fixed):   0%|          | 0/3722 [00:00<?, ?it/s]

[ERROR] Embedding failed: module 'google.genai' has no attribute 'embed_content'
[ERROR] Embedding failed: module 'google.genai' has no attribute 'embed_content'
[ERROR] Embedding failed: module 'google.genai' has no attribute 'embed_content'
[ERROR] Embedding failed: module 'google.genai' has no attribute 'embed_content'
[ERROR] Embedding failed: module 'google.genai' has no attribute 'embed_content'
[ERROR] Embedding failed: module 'google.genai' has no attribute 'embed_content'
[ERROR] Embedding failed: module 'google.genai' has no attribute 'embed_content'
[ERROR] Embedding failed: module 'google.genai' has no attribute 'embed_content'
[ERROR] Embedding failed: module 'google.genai' has no attribute 'embed_content'
[ERROR] Embedding failed: module 'google.genai' has no attribute 'embed_content'
[ERROR] Embedding failed: module 'google.genai' has no attribute 'embed_content'
[ERROR] Embedding failed: module 'google.genai' has no attribute 'embed_content'
[ERROR] Embedding failed: mo

Embedding (struct):   0%|          | 0/11696 [00:00<?, ?it/s]

[ERROR] Embedding failed: module 'google.genai' has no attribute 'embed_content'
[ERROR] Embedding failed: module 'google.genai' has no attribute 'embed_content'
[ERROR] Embedding failed: module 'google.genai' has no attribute 'embed_content'
[ERROR] Embedding failed: module 'google.genai' has no attribute 'embed_content'
[ERROR] Embedding failed: module 'google.genai' has no attribute 'embed_content'
[ERROR] Embedding failed: module 'google.genai' has no attribute 'embed_content'
[ERROR] Embedding failed: module 'google.genai' has no attribute 'embed_content'
[ERROR] Embedding failed: module 'google.genai' has no attribute 'embed_content'
[ERROR] Embedding failed: module 'google.genai' has no attribute 'embed_content'
[ERROR] Embedding failed: module 'google.genai' has no attribute 'embed_content'
[ERROR] Embedding failed: module 'google.genai' has no attribute 'embed_content'
[ERROR] Embedding failed: module 'google.genai' has no attribute 'embed_content'
[ERROR] Embedding failed: mo

## Kaynakça

1. Pinecone. (2023). Chunking Strategies for LLM Applications. Erişim adresi: https://www.pinecone.io/learn/chunking-strategies/
2. Karpukhin, V., Oğuz, B., Min, S., Lewis, P., Wu, L., Edunov, S., Chen, D., & Yih, W. (2020). Dense Passage Retrieval for Open-Domain Question Answering. ArXiv. https://arxiv.org/abs/2004.04906
3. Li, X., Dong, K., Lee, Y. Q., Xia, W., Yin, Y., Zhang, H., Liu, Y., Wang, Y., & Tang, R. (2024). CoIR: A Comprehensive Benchmark for Code Information Retrieval Models. ArXiv. https://arxiv.org/abs/2407.02883
4. Google DeepMind. (2024). Gemini: A Family of Highly Capable Multimodal Models.
5. Zheng, L., Chiang, W.-L., Sheng, Y., Zhuang, S., Wu, Z., Zhuang, Y., Lin, Z., Li, Z., Li, D., Xing, E. P., Zhang, H., Gonzalez, J. E., & Stoica, I. (2023). Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena. arXiv. https://doi.org/10.48550/arXiv.2306.05685
6. Sanders, T., & Heaton, M. (2022, June 10). Question answering using embeddings-based search. OpenAI. https://developers.openai.com/cookbook/examples/question_answering_using_embeddings
7. OpenAI. (n.d.). Evaluation best practices. OpenAI Platform. Retrieved January 21, 2026, from https://platform.openai.com/docs/guides/evaluation-best-practices